# 📊 Exploración Completa — Credit Risk Dataset

Notebook interactivo equivalente al **Stage 2** del pipeline (`preprocessing.py → EDA functions`).

Permite explorar el dataset limpio de forma visual e interactiva antes de ejecutar el pipeline completo.

**Dataset**: [Credit Risk Dataset — Kaggle](https://www.kaggle.com/datasets/laotse/credit-risk-dataset)  
**Prerequisito**: Haber ejecutado el Stage 1 del pipeline (`run_pipeline.py`) o el script de limpieza para generar `data/clean/credit_risk_clean.csv`.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Make the credit_risk package importable from the notebook
PROJECT_ROOT = Path().resolve().parent
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from credit_risk.config import Paths

sns.set_theme(style="whitegrid", palette="muted")
%matplotlib inline
print(f"Project root: {PROJECT_ROOT}")
print(f"Clean data path: {Paths.clean}")

## 1. Carga de datos

In [ ]:
df = pd.read_csv(Paths.clean)
print(f"Shape: {df.shape}")
df.head()

In [ ]:
df.info()

## 2. Estadísticas descriptivas

In [ ]:
df.describe(include="all").T

## 3. Valores faltantes

In [ ]:
missing = df.isna().sum().to_frame(name="count")
missing["pct"] = (df.isna().mean() * 100).round(2)
missing[missing["count"] > 0]

## 4. Distribución del target — `loan_status`

- **0** = No default (cumple sus pagos)
- **1** = Default (incumple)

In [ ]:
dist = df["loan_status"].value_counts(normalize=True).rename({0: "No Default", 1: "Default"})
print(dist.to_string())

fig, ax = plt.subplots(figsize=(5, 4))
dist.plot(kind="bar", ax=ax, color=["steelblue", "tomato"])
ax.set_title("Distribución del target — loan_status", fontweight="bold")
ax.set_ylabel("Proporción")
ax.set_xlabel("")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 5. Distribuciones de variables numéricas

In [ ]:
num_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
print(f"Columnas numéricas: {num_cols}")

n = len(num_cols)
ncols = 3
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(15, nrows * 4))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df[col].dropna(), bins=40, color="steelblue", alpha=0.8, edgecolor="white")
    axes[i].set_title(col, fontweight="bold")
    axes[i].set_ylabel("Frecuencia")

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Distribuciones de variables numéricas", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## 6. Ingreso por estado del crédito (Test visual)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Boxplot
df.boxplot(column="person_income", by="loan_status", ax=axes[0])
axes[0].set_title("Ingreso por estado del crédito")
axes[0].set_xlabel("loan_status (0=No default, 1=Default)")
axes[0].set_ylabel("Ingreso")
plt.sca(axes[0])
plt.title("Boxplot — Ingreso por loan_status")

# Barplot de medias
medias = df.groupby("loan_status")["person_income"].mean()
medias.index = ["No Default", "Default"]
medias.plot(kind="bar", ax=axes[1], color=["steelblue", "tomato"])
axes[1].set_title("Ingreso promedio por estado del crédito", fontweight="bold")
axes[1].set_ylabel("Ingreso promedio")
axes[1].set_xlabel("")
plt.sca(axes[1])
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

print(f"Ingreso medio — No Default: ${medias['No Default']:,.0f}")
print(f"Ingreso medio — Default:    ${medias['Default']:,.0f}")

## 7. Matriz de correlación

In [ ]:
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    ax=ax,
    square=True,
)
ax.set_title("Matriz de correlación — variables numéricas", fontweight="bold", fontsize=13)
plt.tight_layout()
plt.show()

## 8. Variables categóricas

In [ ]:
cat_cols = ["person_home_ownership", "loan_intent", "loan_grade"]

fig, axes = plt.subplots(1, len(cat_cols), figsize=(16, 5))

for ax, col in zip(axes, cat_cols):
    counts = df[col].value_counts()
    counts.plot(kind="bar", ax=ax, color="steelblue", alpha=0.85)
    ax.set_title(col, fontweight="bold")
    ax.set_ylabel("Frecuencia")
    ax.set_xlabel("")
    plt.sca(ax)
    plt.xticks(rotation=30, ha="right")

plt.suptitle("Distribución de variables categóricas", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 9. Resumen EDA

In [ ]:
summary = pd.DataFrame({
    "columna": df.columns,
    "tipo": df.dtypes.astype(str).values,
    "nulos": df.isna().sum().values,
    "pct_nulos": (df.isna().mean() * 100).round(2).values,
    "únicos": df.nunique().values,
})
summary